In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F

#### Passo 2.1 - Definir a população e a granularidade final

Objetivo: Formalizar que a ABT terá uma linha por cliente da application_train.
Por que isso importa em crédito: Toda feature histórica precisa ser reduzida a esse mesmo nível antes de entrar no modelo de application score.


In [0]:
application = spark.table(f"{SILVER}.application_train")
bureau = spark.table(f"{SILVER}.bureau")
previous = spark.table(f"{SILVER}.previous_application")
installments = spark.table(f"{SILVER}.installments_payments")

In [0]:
checks = [ 
("count", application.count()),
("distinct", application.select("SK_ID_CURR").distinct().count())
]

display(spark.createDataFrame(
    checks,
    ["#", "number"]
))

#### Passo 2.2 - Criar features da solicitação atual

Objetivo: Transformar campos brutos em razões com leitura de capacidade e comprometimento.
Por que isso importa em crédito: Relações como crédito/renda são mais comparáveis entre clientes do que valores absolutos isolados.

credit_income_ratio fala mais de alavancagem/tamanho da exposição.
annuity_income_ratio fala mais diretamente de capacidade de pagamento/comprometimento da renda.


In [0]:
application_features = (
    application
    .select(
        "SK_ID_CURR", "TARGET",
        "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY",
        "DAYS_BIRTH", "DAYS_EMPLOYED_CLEAN",
        "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
        "days_employed_anomaly"
    )
    .withColumn("age_years", -F.col("DAYS_BIRTH") / F.lit(365.25))
    .withColumn(
        "credit_income_ratio",
        F.when(
            F.col("AMT_INCOME_TOTAL") > 0,
            F.col("AMT_CREDIT") / F.col("AMT_INCOME_TOTAL")
        )
    )
    .withColumn(
        "annuity_income_ratio",
        F.when(
            F.col("AMT_INCOME_TOTAL") > 0,
            F.col("AMT_ANNUITY") / F.col("AMT_INCOME_TOTAL")
        )
    )
    .withColumn(
        "employment_years", -F.col("DAYS_EMPLOYED_CLEAN") / F.lit(365.25)

    )
)   

In [0]:
application_features.select("age_years", "credit_income_ratio", "annuity_income_ratio", "employment_years").show(10)

In [0]:
(
    application_features
    .select('SK_ID_CURR', 'credit_income_ratio')
    .join(
        bureau.select(
            'SK_ID_CURR',
            'AMT_CREDIT_SUM',
            'AMT_CREDIT_SUM_DEBT'
        ),
        "SK_ID_CURR",
        "left"
)
).show(10)

In [0]:
percentis = application_features.approxQuantile(
    "credit_income_ratio",
    [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99],
    0.01
)

print(percentis)

#### Passo 2.3 - Fechar a relação SK_ID_PREV entre previous e installments

**Objetivo**: Confirmar que um contrato anterior aponta para um único cliente.
**Por que isso importa em crédito**: Se SK_ID_PREV estiver associado a mais de um SK_ID_CURR, o agregado por contrato não poderá ser levado ao cliente com segurança.


In [0]:
prev_key_check = (
    previous
    .groupBy("SK_ID_PREV")
    .agg(
        F.countDistinct("SK_ID_CURR").alias("curr_ids"))
    .filter(
        F.col("curr_ids") > 1)
    )
inst_key_check = (
    installments
    .groupBy("SK_ID_PREV")
    .agg(
        F.countDistinct("SK_ID_CURR").alias("curr_ids")
    )
    .filter(
        F.col("curr_ids") > 1
    )
)

print(f"SK_ID_PREV ambiguos em previous:{prev_key_check.count()}")
print(f"SK_ID_PREV ambiguos em installments:{inst_key_check.count()}")

In [0]:
previous_ids = previous.select("SK_ID_PREV").distinct()
installments_ids = installments.select("SK_ID_PREV").distinct()

print(
    f"previous com installments {previous_ids.join(installments_ids, "SK_ID_PREV", "left_semi").count():,}"
)
print(
    f"previous sem installments {previous_ids.join(installments_ids, 'SK_ID_PREV', 'left_anti').count():,}"
)

In [0]:
previous.count()

#### Passo 2.4 - Criar atraso e insuficiência de pagamento por registro


DESAFIO GUIADO - tente montar as regras
Para cada registro de installments, crie dias de atraso, valor não pago, flag de atraso, flags 30/60/90+ DPD e razão pago/devido. Antecipação não deve virar atraso negativo.
> Pistas permitidas: DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT, greatest, coalesce, when.

>No installments_payments:

SK_ID_PREV → identificador da operação/crédito anterior.
SK_ID_CURR → identificador do cliente.
NUM_INSTALMENT_VERSION → versão do plano de parcelas.
NUM_INSTALMENT_NUMBER → número da parcela.
DAYS_INSTALMENT → dia em que a parcela deveria ser paga, em dias relativos à solicitação atual.
DAYS_ENTRY_PAYMENT → dia em que o pagamento realmente aconteceu.
AMT_INSTALMENT → valor que deveria ser pago.
AMT_PAYMENT → valor que efetivamente foi pago.

In [0]:
(
    installments
    .select("*")
    .filter(
        F.col("SK_ID_CURR") == '161674'
    )

).show()

In [0]:
# dias de atraso
installments_features = (
    installments
    .select('*')
    .withColumn(
        'OVERDUE_DAYS',
        F.when(F.col('DAYS_ENTRY_PAYMENT') - F.col("DAYS_INSTALMENT") > 0,
               F.col('DAYS_ENTRY_PAYMENT') - F.col("DAYS_INSTALMENT")
    ).otherwise(0)
 )
)

In [0]:
(
    installments
    .select('*')
    .filter(
        F.col('AMT_PAYMENT').isNull()
    )
).show()

In [0]:
installments_features = (
    installments_features
    .select('*')
    .withColumn(
        'IS_PAYMENT_OVERDUE',
        F.when((F.col("AMT_PAYMENT").isNull()) & F.col("DAYS_ENTRY_PAYMENT").isNull(), True).otherwise(False)

    )
)

In [0]:
installments_features.show()

In [0]:
installments_features.printSchema()

In [0]:
installments_features = (
    installments_features
    .select('*')
    .withColumn(
        'DELAY_RANGE',
        F.when((F.col('OVERDUE_DAYS') > 0) & (F.col('OVERDUE_DAYS') <= 30), "0 - 30")
        .when((F.col('OVERDUE_DAYS') > 30) & (F.col('OVERDUE_DAYS') <= 60), '31 - 60')
        .when((F.col('OVERDUE_DAYS') > 60) & (F.col('OVERDUE_DAYS') <= 90), '61 - 90')
        .when((F.col('OVERDUE_DAYS') > 90) & (F.col('OVERDUE_DAYS') <= 120), '91- 120')
        .when((F.col('OVERDUE_DAYS') > 120), '> 120')
        .otherwise(F.lit('UP TO DATE'))
    )
)

In [0]:
installments_features.show()

In [0]:
(
    installments_features
    .select('*')
    .filter(
        F.col('AMT_PAYMENT') < F.col('AMT_INSTALMENT')
    )
    .show()
)

In [0]:
installments_features = (
    installments_features
    .select('*')
    .withColumn(
        'PAYMENT_RATIO',
        F.col('AMT_PAYMENT') / F.col('AMT_INSTALMENT')

    )
)

In [0]:
(
    installments_features
    .select('*')
    .filter(
        F.col('PAYMENT_RATIO') > 1
    )
).show()

In [0]:
(
    installments_features
    .select('*')
    .filter(
        F.col("SK_ID_PREV") == '2000031'
    )
    .orderBy('NUM_INSTALMENT_NUMBER')
    .show(100)
)

In [0]:
(
    installments_features
    .select('*')
    .groupBy('SK_ID_PREV')
    .agg(
        F.sum("AMT_INSTALMENT")
    )
    .filter(
        F.col('SK_ID_PREV') == '2000031')
).show()

In [0]:
(
    previous
    .select('*')
    .filter(
        F.col('SK_ID_PREV') == '2000031'
    )
).show()

In [0]:
installment_check = (
    installments
    .groupBy(
        "SK_ID_PREV",
        "NUM_INSTALMENT_NUMBER"
    )
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("AMT_PAYMENT").alias("payment_values"),
        F.sum("AMT_INSTALMENT").alias("total_due"),
        F.max("AMT_PAYMENT").alias("payment")
    )
)

installment_check.filter(
    F.col("rows") > 1
).show()

#### Passo 2.5 - Agregar installments por operação

Objetivo: Reduzir várias parcelas/pagamentos para uma linha por SK_ID_PREV.
Por que isso importa em crédito: Só depois dessa redução o resumo pode ser juntado à previous_application sem multiplicar propostas.


In [0]:
installments_features_prev = (
    installments_features
    .groupBy('SK_ID_PREV')
    .agg(
        F.first('SK_ID_CURR', ignorenulls = True).alias('SK_ID_CURR_INST'),
        F.count('*'),
        F.sum('AMT_INSTALMENT').alias('AMT_INSTALMENT_SUM'),
        F.sum('AMT_PAYMENT').alias('AMT_PAYMENT_SUM'),
        F.avg('OVERDUE_DAYS').alias('AVG_OVERDUE_DAYS'),
        F.max('OVERDUE_DAYS').alias('MAX_OVERDUE_DAYS'),
        F.sum(
            F.when(F.col('IS_PAYMENT_OVERDUE') == True, 1).otherwise(0)).alias('PAYMENT_OVERDUE_COUNT')
        
        )
    
)

In [0]:
installments_features.printSchema

In [0]:
installments_features_prev.show()

In [0]:
(
    installments_features_prev
    .select('*')
    .filter(
        F.col('AMT_PAYMENT_SUM') > F.col('AMT_INSTALMENT_SUM')
    )

).show()

In [0]:
installments_features_prev.count() == (
    installments_features_prev
    .select("SK_ID_PREV")
    .distinct()
    .count()
)

#### Passo 2.6 - Juntar previous com o resumo de installments

In [0]:
previous.show()

In [0]:
previous_enriched = (

    previous
    .select('*')
    .join(
        installments_features_prev,
        "SK_ID_PREV",
        "left"
        )
)

In [0]:
print("Antes:", previous.count())
print("Depois:", previous_enriched.count())

print(
    "SK_ID_PREV depois:",
    previous_enriched.select("SK_ID_PREV").distinct().count()
)

#### Passo 2.7 - Agregar Propostas e Pagamentos por Cliente

In [0]:
previous_enriched.show()

In [0]:
previous_features = (
    previous_enriched
    .groupBy('SK_ID_CURR')
    .agg(
        F.countDistinct('SK_ID_PREV').alias('NUMBER_PROPOSAL'),
        F.sum(
            F.when(F.col('NAME_CONTRACT_STATUS') == 'Approved', 1).otherwise(0)).alias('APPROVED_COUNT'),
        F.sum(
            F.when(F.col('NAME_CONTRACT_STATUS') == 'Refused', 1).otherwise(0)).alias('REFUSED_COUNT'),
        F.avg(
            F.when(F.col('NAME_CONTRACT_STATUS') == 'Approved', F.col("AMT_CREDIT"))).alias('AMT_CREDIT_AVG'),
        (F.col('REFUSED_COUNT') / F.col('NUMBER_PROPOSAL')).alias('REFUSED_RATIO')
    )
)

In [0]:
previous_features.show()

In [0]:
(
    previous
    .select('*')
    .filter(
        F.col('SK_ID_CURR') == 271877
    )

).show()

In [0]:
(
    previous
    .select('NAME_CONTRACT_STATUS').distinct().show()
)